# Practical 05 – Multiple and Polynomial Regression

**Course:** ML6402 – Machine Learning  
**Instructor:** Dr. Ahmed Sabbah  
**University:** Al-Quds Open University


This practical covers categorical encoding, multiple regression, evaluation, and polynomial regression.

In [ ]:
from pathlib import Path
import pandas as pd

def load_course_csv(filename):
    candidates = [Path("data") / filename, Path("Lecture05_Multiple_and_Polynomial_Regression") / "data" / filename]
    local_path = next((p for p in candidates if p.exists()), None)
    url = f"https://raw.githubusercontent.com/asabbah44/ML6402-Machine-Learning-Practicals/main/Lecture05_Multiple_and_Polynomial_Regression/data/{filename}"
    source = local_path if local_path else url
    print("Loaded from:", source)
    return pd.read_csv(source)

In [ ]:
df = load_course_csv("50_AdAgency.csv")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df.drop(columns=["Profit"])
y = df["Profit"]

preprocess = ColumnTransformer([
    ("city", OneHotEncoder(drop="first", handle_unknown="ignore"), ["City"])
], remainder="passthrough")

model = Pipeline([
    ("preprocess", preprocess),
    ("regressor", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("MAE :", mean_absolute_error(y_test, pred))
print("RMSE:", mean_squared_error(y_test, pred) ** 0.5)
print("R²  :", r2_score(y_test, pred))

## Interpret the fitted multiple-regression model

After fitting the pipeline, the regression model contains one coefficient for every **transformed input feature**.

Because `City` is categorical, `OneHotEncoder` first creates dummy variables. Therefore, we should not read coefficients directly without also retrieving the transformed feature names.

The steps are:

1. get the final feature names produced by the preprocessing step;
2. get the coefficients learned by `LinearRegression`;
3. combine them into a table;
4. sort by absolute coefficient size to make inspection easier.

**Important:** a large coefficient is not automatically the "most important" feature because coefficients depend on the units and scale of each feature. The table is mainly used here to connect the fitted model with the regression equation and interpret the direction of each coefficient.

In [ ]:
feature_names = (
    model.named_steps["preprocess"]
         .get_feature_names_out()
)

coef = model.named_steps["regressor"].coef_
intercept = model.named_steps["regressor"].intercept_

coef_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coef
})

print("Intercept (b0):", intercept)
display(
    coef_table.sort_values(
        "coefficient",
        key=abs,
        ascending=False
    )
)

### How to read the output

For a numerical feature, a positive coefficient means that the predicted profit tends to increase as that feature increases, **while the other predictors are held constant**. A negative coefficient means the predicted profit tends to decrease.

For the encoded `City` variables, the coefficient represents the difference relative to the category that was dropped by `OneHotEncoder(drop="first")`.

The intercept `b₀` is the model's baseline prediction when all numerical predictors are 0 and the categorical variable is at its reference category.

The feature names may look like:

```text
city__City_Delhi
city__City_Mumbai
remainder__Print Media Expenses
remainder__Social Media Expenses
remainder__Outdoor Ad Expenses
```

The prefixes such as `city__` and `remainder__` are added by Scikit-Learn's `ColumnTransformer` so we can identify where each transformed feature came from.

# Part B – Polynomial Regression

The lecture uses Bhatia's **Position Level → Salary** case study to show why a straight line may be too simple.

In this section we will:

1. inspect the original relationship;
2. see what polynomial features actually create;
3. fit degrees 2, 3, and 4;
4. compare linear and polynomial regression;
5. predict salary for a new rank;
6. inspect an excessively high polynomial degree;
7. choose model complexity using validation evidence.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

salary_df = load_course_csv("Position_Salaries.csv")
salary_df

## Inspect the relationship first

Before selecting a model, plot the data.

The salary increase accelerates at higher position levels, so a single constant slope may not represent the whole relationship well.

In [ ]:
X_poly = salary_df[["Rank"]]
y_poly = salary_df["Salary"]

plt.scatter(X_poly["Rank"], y_poly)
plt.xlabel("Position level / Rank")
plt.ylabel("Salary")
plt.title("Position Level vs Salary")
plt.show()

## What does `PolynomialFeatures` create?

Polynomial regression does not make `LinearRegression` itself nonlinear.

Instead, it creates new input columns.

For example:

- degree 2: `x → [x, x²]`
- degree 3: `x → [x, x², x³]`

The linear regression model then learns one coefficient for each transformed feature.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

example_x = np.array([[2], [3], [5]], dtype=float)

poly2_demo = PolynomialFeatures(degree=2, include_bias=False)
poly3_demo = PolynomialFeatures(degree=3, include_bias=False)

print("Original x:")
print(example_x.ravel())

print("\nDegree-2 representation:")
print(poly2_demo.fit_transform(example_x))

print("\nDegree-3 representation:")
print(poly3_demo.fit_transform(example_x))

## Fit polynomial degrees 2, 3, and 4

This reproduces the progression discussed in the lecture and Bhatia example.

As degree increases, the model becomes more flexible. Training error will often decrease, but that alone does not prove better generalization.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

degrees = [2, 3, 4]
poly_models = {}

grid = pd.DataFrame({
    "Rank": np.linspace(X_poly["Rank"].min(), X_poly["Rank"].max(), 300)
})

plt.scatter(X_poly["Rank"], y_poly, label="Observed data")

for degree in degrees:
    poly_model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression()
    )
    poly_model.fit(X_poly, y_poly)
    poly_models[degree] = poly_model

    y_fit = poly_model.predict(X_poly)

    print(f"Degree {degree}")
    print("  MAE :", round(mean_absolute_error(y_poly, y_fit), 2))
    print("  RMSE:", round(mean_squared_error(y_poly, y_fit) ** 0.5, 2))
    print("  R²  :", round(r2_score(y_poly, y_fit), 4))

    plt.plot(grid["Rank"], poly_model.predict(grid), label=f"Degree {degree}")

plt.xlabel("Position level / Rank")
plt.ylabel("Salary")
plt.title("Polynomial Regression with Different Degrees")
plt.legend()
plt.show()

## Linear regression vs degree-4 polynomial regression

This comparison shows why the polynomial model can follow the accelerating salary pattern better than a straight line.

Remember: better training fit is only diagnostic. The final degree should be justified with validation evidence.

In [ ]:
linear_salary_model = LinearRegression()
linear_salary_model.fit(X_poly, y_poly)

degree4_model = poly_models[4]

plt.scatter(X_poly["Rank"], y_poly, label="Observed data")
plt.plot(grid["Rank"], linear_salary_model.predict(grid), label="Linear regression")
plt.plot(grid["Rank"], degree4_model.predict(grid), label="Degree-4 polynomial")

plt.xlabel("Position level / Rank")
plt.ylabel("Salary")
plt.title("Linear vs Polynomial Regression")
plt.legend()
plt.show()

linear_pred = linear_salary_model.predict(X_poly)
poly4_pred = degree4_model.predict(X_poly)

print("Linear RMSE :", mean_squared_error(y_poly, linear_pred) ** 0.5)
print("Degree-4 RMSE:", mean_squared_error(y_poly, poly4_pred) ** 0.5)

## Predict a new value

The Bhatia example predicts salary at a position level around **7.5**.

Because the polynomial transformation is inside the pipeline, the new input is transformed in exactly the same way as the training data.

In [ ]:
new_rank = pd.DataFrame({"Rank": [7.5]})
predicted_salary = degree4_model.predict(new_rank)[0]

print("Predicted salary for rank 7.5:", round(predicted_salary, 2))

## What happens if the degree is too high?

Bhatia explicitly discusses a very high degree such as 50.

A highly flexible polynomial can fit the training observations extremely closely but may produce unstable or unrealistic behavior between observations.

This is a direct introduction to **overfitting**.

In [ ]:
high_degree_model = make_pipeline(
    PolynomialFeatures(degree=50, include_bias=False),
    LinearRegression()
)

high_degree_model.fit(X_poly, y_poly)

plt.scatter(X_poly["Rank"], y_poly, label="Observed data")
plt.plot(grid["Rank"], high_degree_model.predict(grid), label="Degree 50")

plt.xlabel("Position level / Rank")
plt.ylabel("Salary")
plt.title("Excessive Polynomial Degree")
plt.legend()
plt.show()

high_degree_train_pred = high_degree_model.predict(X_poly)
print(
    "Degree-50 training RMSE:",
    mean_squared_error(y_poly, high_degree_train_pred) ** 0.5
)

## Source Chapter Coverage Note

This practical now focuses on the worked examples that are actually present in the author's multiple/polynomial-regression chapter:

- multiple linear regression with `50_AdAgency.csv`;
- categorical encoding and dummy-variable handling;
- train/test split, fitting, prediction, and regression metrics;
- coefficient inspection;
- polynomial regression with `Position_Salaries.csv`;
- degrees 2, 3, and 4;
- prediction at rank 7.5;
- excessive degree (degree 50);
- comparison of linear and polynomial fits.

The author's chemistry-lab polynomial example references `Chemistry_Lab.csv`, but that dataset is not present in the source repository, so it is not reproduced here with invented data.